In [ ]:
import re
import sys
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import COLUMN_PHYSICAL_RANGES

RAW_PATH = PROJECT_ROOT / "data" / "raw" / "sensor_data_raw2.csv"
PROCESSED_PATH = PROJECT_ROOT / "data" / "processed" / "clean_data.parquet"

print("Project root:", PROJECT_ROOT)
print("Raw file:", RAW_PATH)
print("Exists:", RAW_PATH.exists())

In [ ]:
df = pd.read_csv(RAW_PATH, sep=";", dtype=str)

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print("Columns:", list(df.columns))
df.head()

In [ ]:
def clean_column_name(raw: str) -> str:
    name = str(raw).strip()
    name = re.sub(r"[\(\)]", "", name)
    return name.strip()


df.columns = [clean_column_name(c) for c in df.columns]

# Fecha / FECHA → FECHA
fecha_cols = [c for c in df.columns if str(c).strip().upper() == "FECHA"]
if not fecha_cols:
    raise ValueError(f"Timestamp column not found. Columns: {list(df.columns)}")
if fecha_cols[0] != "FECHA":
    df = df.rename(columns={fecha_cols[0]: "FECHA"})

# Drop fully empty rows
df = df.dropna(how="all").reset_index(drop=True)

print("Cleaned columns:", list(df.columns))
df.head(2)

In [ ]:
raw_fecha = df["FECHA"].astype(str)

# Extract timezone offset hours if present (+1H, +2H); else 0
offsets = raw_fecha.str.extract(r"\+(\d+)H", expand=False).fillna("0").astype(int)

# Remove suffix, then parse (dayfirst for DD/MM/YYYY)
cleaned_fecha = raw_fecha.str.replace(r"\s*\+\d+H.*$", "", regex=True).str.strip()
df["FECHA"] = pd.to_datetime(cleaned_fecha, errors="coerce", dayfirst=True)

# Convert to UTC when an offset was present
df["FECHA"] = df["FECHA"] - pd.to_timedelta(offsets, unit="h")

n_bad = df["FECHA"].isna().sum()
print(f"Unparsed timestamps: {n_bad}")

df = df.dropna(subset=["FECHA"]).reset_index(drop=True)
print(f"Rows after dropping bad dates: {len(df):,}")
print(f"Time range: {df['FECHA'].min()} → {df['FECHA'].max()}")
df[["FECHA"]].head()

In [ ]:
offset_counts = offsets.value_counts().sort_index()
print("Timezone offset counts (hours subtracted to get UTC):")
print(offset_counts.to_string())

if (offsets == 0).all():
    print("\nNo +NH suffix found → timestamps kept as parsed (no UTC shift).")
else:
    print("\nOffsets detected → timestamps converted to UTC by subtracting the offset.")

In [ ]:
sensor_cols = [c for c in df.columns if c != "FECHA"]

for col in sensor_cols:
    if df[col].dtype == object:
        df[col] = (
            df[col]
            .astype(str)
            .str.replace(",", ".", regex=False)
            .replace({"nan": "NaN", "None": "NaN", "": "NaN"})
        )
    df[col] = pd.to_numeric(df[col], errors="coerce")

print(df.dtypes)
df.head()

In [ ]:
was_sorted = df["FECHA"].is_monotonic_increasing
print(f"Already sorted: {was_sorted}")

df = df.sort_values("FECHA").reset_index(drop=True)

print(f"Sorted now: {df['FECHA'].is_monotonic_increasing}")
print(f"First: {df['FECHA'].iloc[0]}")
print(f"Last:  {df['FECHA'].iloc[-1]}")

In [ ]:
n_before = len(df)
n_dupes = df["FECHA"].duplicated().sum()
print(f"Duplicate timestamps: {n_dupes}")

if n_dupes:
    display(df.loc[df["FECHA"].duplicated(keep=False)].head(10))

df = df.drop_duplicates(subset=["FECHA"], keep="first").reset_index(drop=True)
print(f"Rows removed: {n_before - len(df)}")
print(f"Rows after dedup: {len(df):,}")

In [ ]:
time_diff = df["FECHA"].diff()
time_diff_seconds = time_diff.dt.total_seconds()

print("Sampling interval (seconds) — summary:")
print(time_diff_seconds.describe())

sampling_summary = (
    time_diff_seconds
    .round(0)
    .value_counts()
    .rename_axis("interval_seconds")
    .reset_index(name="count")
)
sampling_summary["percentage"] = (
    sampling_summary["count"] / sampling_summary["count"].sum() * 100
).round(2)

print("\nMost common intervals:")
sampling_summary.head(35)

In [ ]:
GAP_THRESHOLD_SECONDS = 60

gap_mask = time_diff_seconds > GAP_THRESHOLD_SECONDS
gaps = df.loc[gap_mask, ["FECHA"]].copy()
gaps["gap_duration"] = time_diff.loc[gap_mask]
gaps["gap_seconds"] = time_diff_seconds.loc[gap_mask]

print(f"Number of gaps > {GAP_THRESHOLD_SECONDS}s: {len(gaps)}")
gaps

In [ ]:
nan_report = pd.DataFrame({
    "null_count": df[sensor_cols].isna().sum(),
    "null_pct": (df[sensor_cols].isna().mean() * 100).round(2),
})
nan_report = nan_report.sort_values("null_count", ascending=False)

print(f"Total rows: {len(df):,}")
print(f"Rows with any NaN: {df[sensor_cols].isna().any(axis=1).sum():,}")
nan_report

In [ ]:
range_rows = []

for col, (lo, hi) in COLUMN_PHYSICAL_RANGES.items():
    if col not in df.columns:
        continue
    series = df[col]
    out_low = (series < lo).sum()
    out_high = (series > hi).sum()
    range_rows.append({
        "column": col,
        "min_allowed": lo,
        "max_allowed": hi,
        "data_min": series.min(),
        "data_max": series.max(),
        "below_range": int(out_low),
        "above_range": int(out_high),
        "out_of_range": int(out_low + out_high),
    })

range_report = pd.DataFrame(range_rows).set_index("column")
print("Physical-range violations:")
range_report

In [ ]:
PROCESSED_PATH.parent.mkdir(parents=True, exist_ok=True)
df.to_parquet(PROCESSED_PATH, index=False, engine="pyarrow")

size_mb = PROCESSED_PATH.stat().st_size / 1e6
print(f"Saved: {PROCESSED_PATH}")
print(f"Size:  {size_mb:.2f} MB")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")

# Round-trip check
df_check = pd.read_parquet(PROCESSED_PATH, engine="pyarrow")
assert df_check.shape == df.shape
print("Reload OK")
df_check.head()